# 01 — Unified dataset (2019-2026): GOES image stack + per-cell features

The single cache builder for the whole project. Task: predict which 50 km CONUS-land cells
flood on CDT day D from the previous day's (D−1) GOES. Replaces the old 01 (raw-band) and
01b (2019-only feature) notebooks with one build over the full **2019-2026** dataset.

## What gets cached, per sample (one CDT day)

1. GOES image stack, shape **(8, 7, 750, 1250)** f16 — for CNN / ConvLSTM encoders.
   Stored at **half resolution** (the native 1500×2500 is downsampled /2 by block-mean,
   ~4 km/px, still mesoscale-resolving) so the full 2019-2026 cache fits on the fast NVMe.
   7 channels: 3 raw bands (b8 upper-WV, b10 low-WV, b14 IR-window) + 2 brightness-
   temperature differences (d10-8 WV gradient, d11-14 cloud phase) + 2 temporal channels
   (3-hourly Δb14, running max cloud-top cooling). Standardized per channel.
2. Per-cell feature grids on the 59×95 land grid, by masked pooling of the **full-res**
   pixels (computed before the image is downsampled, so no signal is lost):
   - GOES cell features, shape (8, 4, R, C): b14_min (coldest pixel), cold-cloud fractions
     (b14 < 235 K, b14 < 220 K), and the cold-cloud area change vs the previous frame.
   - GLM cell features, shape (8, 2, R, C): per 3-hour bin — flash count, total flash energy.
   - Daily summaries, shape (3, R, C): daily min b14, daily min b8, daily GLM flash count.

Plus per-frame lead time (8,) in hours before day D / 24, and the label (R, C), a 0/1
observed-flood map (Groundsource ∪ NCEI storm events).

The feature set was trimmed from 16 image + 43 per-cell channels after measuring univariate
flood signal and inter-feature correlation on the 2019 cache: redundant IR-window bands
(b11/b13/b15 ≈ b14), the per-cell means (duplicate the CNN's mean-pool), and the collapsed
GLM dimensions / buggy area field were all dropped.

## Split (canonical, temporal)

train **2019-2024**, validate **2025**, test **2026**. (2026 is a partial, still-lagging
year, so its label density is lower than the training years.)

Days are CDT (UTC−5); GOES files stay UTC. One CDT day's 8 frames span two UTC folders
(06–21 UTC of D−1 plus 00 and 03 UTC of D). Storm events shift −5 h to CDT; Groundsource
dates are day-resolution and used as-is.

Pipeline: grid + pooling index → labels → pixel stack → per-cell pooling → GLM → daily
summaries → standardize → index/temporal-split → materialize cache → summarize.

## 0. Config and imports

Feature definitions and thresholds live here; the grid, the year span (YEARS), and the
cache location (CACHE_DIR) come from the repo-root config.py — the single source of truth
the trainers also read. The cache is `cache/goes_features_2019_2026` on the root NVMe.

In [ ]:
import json
import sys
import time
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd

# repo root + notebooks/model on path (for config.py and gridindex.py)
ROOT = Path.cwd()
while not (ROOT / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "notebooks" / "model"))
from config import (CACHE_DIR, DATA_DIR, GLM_DIR, IMG_H, IMG_W, T_FRAMES, YEARS,
                    UNIFIED_PARQUET, build_grid_cells, grid_transform)
from gridindex import build_pix2cell

# --- full dataset 2019-2026, unified cache on the root NVMe (from config) ---
STATS_PATH = CACHE_DIR / "img_channel_stats.json"

# temporal split: train <= TRAIN_MAX, val == VAL_YEAR, test == TEST_YEAR
TRAIN_MAX, VAL_YEAR, TEST_YEAR = 2024, 2025, 2026

# stored image resolution: native ABI is 1500x2500; store at 1/DOWNSAMPLE (block-mean) so
# the full 2019-2026 image cache (~284 GB) fits on the fast NVMe. Per-cell features are
# pooled from the FULL-res pixels first, so only the CNN's image input is coarsened.
DOWNSAMPLE = 2
IMG_H_DS, IMG_W_DS = IMG_H // DOWNSAMPLE, IMG_W // DOWNSAMPLE      # 750 x 1250 stored
assert IMG_H % DOWNSAMPLE == 0 and IMG_W % DOWNSAMPLE == 0

# image channels (7): 3 raw bands + 2 BT differences + 2 temporal-change channels.
# Trimmed from 16 after measuring redundancy on the cached data: b11/b13/b15 duplicate b14
# (r > 0.994) and b9 duplicates b8 (r = 0.98), so only b8/b10/b14 are kept raw; d14-10 ≈ b14,
# d14-15 had ~no flood signal, d9-10 ≈ -d10-8, and dt_b8/dt_b10 ≈ dt_b14 — all dropped.
RAW_BANDS = (8, 10, 14, 11)            # bands read per file (b11 is only used to form d11-14)
RAW_KEEP = (8, 10, 14)                 # raw bands kept as their own image channels
DIFF_PAIRS = ((10, 8), (11, 14))       # d10-8 (low-WV gradient), d11-14 (cloud phase / ice)
IMG_CH_NAMES = ([f"b{b}" for b in RAW_KEEP]
                + [f"d{a}-{b}" for a, b in DIFF_PAIRS]
                + ["dt_b14_3h", "maxcool_b14"])
N_IMG_CH = len(IMG_CH_NAMES)            # 7

# cold-cloud / deep-convection thresholds (Kelvin)
CC235, CC220 = 235.0, 220.0            # b14 cold-cloud-top fractions (convective coverage/depth)

print(f"years {min(YEARS)}-{max(YEARS)}  |  {N_IMG_CH} image channels (stored "
      f"{IMG_H_DS}x{IMG_W_DS}, /{DOWNSAMPLE}):")
print("  ", IMG_CH_NAMES)
print(f"split: train <={TRAIN_MAX}  val {VAL_YEAR}  test {TEST_YEAR}")
print(f"cache -> {CACHE_DIR}")

## 1. Output grid and the masked-pooling index

The label space and the per-cell features both live on the same fixed grid of 50 km CONUS
cells (config.build_grid_cells, EPSG:5070). The pixel→cell map (gridindex.build_pix2cell)
assigns every full-resolution ABI pixel to its cell; we turn it into a sorted index so
each cell's pixels form a contiguous block. That gives true masked pooling — every
aggregation (mean, min, fraction) is computed over exactly the valid pixels that fall in a
cell, never an image resize.

In [ ]:
cells, GRID_R, GRID_C, land_mask = build_grid_cells()
p2c, _, _, _ = build_pix2cell()                       # (1500,2500) -> flat cell id, -1 off-grid
N_CELL = GRID_R * GRID_C
n_land = int(land_mask.sum())

# group valid pixels by cell: VIDX gathers pixel values into cell-contiguous blocks
_flat = p2c.ravel()
_vidx = np.flatnonzero(_flat >= 0)
_cell = _flat[_vidx].astype(np.int64)
_order = np.argsort(_cell, kind="stable")
VIDX = _vidx[_order]                                   # pixel linear indices, grouped by cell
CELLG = _cell[_order]                                  # cell id per gathered pixel (ascending)
print(f"grid {GRID_R}x{GRID_C}  |  land cells {n_land}  |  pooled pixels {len(VIDX):,}")


def pool_means(stack):
    """(C,H,W) physical -> (C,GRID_R,GRID_C): masked mean of each channel over a cell's pixels."""
    C = stack.shape[0]
    v = stack.reshape(C, -1)[:, VIDX]
    fin = np.isfinite(v)
    v0 = np.where(fin, v, 0.0)
    out = np.full((C, N_CELL), np.nan, np.float32)
    for c in range(C):
        s = np.bincount(CELLG, weights=v0[c], minlength=N_CELL)
        n = np.bincount(CELLG, weights=fin[c].astype(np.float64), minlength=N_CELL)
        nz = n > 0
        out[c, nz] = (s[nz] / n[nz]).astype(np.float32)
    return out.reshape(C, GRID_R, GRID_C)


def pool_frac(a, thr):
    """Fraction of a cell's valid pixels with value < thr."""
    v = a.ravel()[VIDX]
    fin = np.isfinite(v)
    below = np.where(fin, (v < thr), 0.0).astype(np.float64)
    s = np.bincount(CELLG, weights=below, minlength=N_CELL)
    n = np.bincount(CELLG, weights=fin.astype(np.float64), minlength=N_CELL)
    out = np.zeros(N_CELL, np.float32)
    nz = n > 0
    out[nz] = (s[nz] / n[nz]).astype(np.float32)
    return out.reshape(GRID_R, GRID_C)


def pool_min(a):
    """Per-cell minimum of a channel (masked)."""
    v = a.ravel()[VIDX].astype(np.float64)
    v = np.where(np.isfinite(v), v, np.inf)
    out = np.full(N_CELL, np.inf)
    np.minimum.at(out, CELLG, v)
    out[~np.isfinite(out)] = np.nan
    return out.reshape(GRID_R, GRID_C).astype(np.float32)

## 2. Labels — observed floods to daily 0/1 cell maps

Over all of 2019-2026. A cell is positive on CDT day D if a Groundsource news-report extent
or an NCEI storm-event footprint intersects it. Storm events shift −5 h to CDT; Groundsource
dates (day-resolution) are used as-is. An event spanning up to 5 CDT days marks each day;
longer collapses to its issue day.

In [ ]:
LABEL_SOURCES = {"groundsource", "storm_event"}
CDT = pd.Timedelta(hours=5)
MAX_SPLIT_DAYS = 5

u = gpd.read_parquet(UNIFIED_PARQUET)
w = u[u["issue_date"].dt.year.isin(YEARS) & u["source"].isin(LABEL_SOURCES)].copy()

shift = pd.Series(pd.Timedelta(0), index=w.index)
shift[w["source"] == "storm_event"] = CDT             # only storm events carry a real UTC time
w["issue_cdt"] = w["issue_date"] - shift
w["expire_cdt"] = w["expire_date"] - shift


def event_days(issue, expire):
    "CDT days an event labels: each day if it spans <= MAX_SPLIT_DAYS, else the issue day."
    days = pd.date_range(issue.normalize(), expire.normalize(), freq="D")
    return days if len(days) <= MAX_SPLIT_DAYS else days[:1]


w["days"] = [event_days(i, e) for i, e in zip(w["issue_cdt"], w["expire_cdt"])]
ev = w.explode("days", ignore_index=True).rename(columns={"days": "label_day"})
ev = ev[ev["label_day"].dt.year.isin(YEARS)]

j = gpd.sjoin(cells, ev[["label_day", "geometry"]], predicate="intersects")
labels = {}
for day, g in j.groupby("label_day"):
    a = np.zeros((GRID_R, GRID_C), np.float32)
    a[g["R"], g["C"]] = 1.0
    labels[day.date()] = a

cnt = w["source"].value_counts()
pos_per_day = np.array([a[land_mask].sum() for a in labels.values()])
print(f"{len(w):,} flood events ({' + '.join(f'{v:,} {k}' for k, v in cnt.items())}) "
      f"-> {len(ev):,} event-days on {len(labels)} CDT days, {min(YEARS)}-{max(YEARS)}")
print(f"flooded land cells/day: median {np.median(pos_per_day):.0f}, max {pos_per_day.max():.0f} "
      f"({np.median(pos_per_day)/n_land:.1%} of land)")
# label days per calendar year
_ly = pd.Series([d.year for d in labels]).value_counts().sort_index()
print("label days/year:", {int(y): int(c) for y, c in _ly.items()})

## 3. GOES file and time helpers

One NetCDF per scan on /mnt/disk4, 8 frames per UTC day (00, 03, …, 21 UTC). goes_cdt_day
returns the 8 frames whose CDT date (scan UTC − 5 h) equals the requested day, globbing the
two UTC folders the CDT day spans.

In [ ]:
def _scan_token(p):
    "The s-token (s{YYYYDDDHHMM...}) from a GOES filename."
    for part in p.name.split("_"):
        if part.startswith("s") and part[1:].isdigit():
            return part
    return p.name


def _scan_dt(p):
    "Scan-start datetime (UTC) parsed from the filename token."
    return datetime.strptime(_scan_token(p)[1:12], "%Y%j%H%M")


def goes_files(d):
    "Sorted GOES NetCDFs in the UTC day folder d (GOES16/19 auto-globbed)."
    pat = f"*/{d.year}/{d.month:02d}/{d.day:02d}/*.nc"
    return sorted(DATA_DIR.glob(pat), key=_scan_token)


def goes_cdt_day(cdt_day):
    "GOES frames whose CDT date (scan UTC - 5 h) equals cdt_day, in scan order."
    nxt = cdt_day + timedelta(days=1)
    cand = sorted(goes_files(cdt_day) + goes_files(nxt), key=_scan_token)
    return [f for f in cand if (_scan_dt(f) - timedelta(hours=5)).date() == cdt_day]


def valid_day(label_day):
    "True if CDT day D-1 has all T_FRAMES GOES frames."
    return len(goes_cdt_day(label_day - timedelta(days=1))) == T_FRAMES


_valid = []                                            # first few valid days (short-circuit)
for d in sorted(labels):
    if valid_day(d):
        _valid.append(d)
        if len(_valid) >= 6:
            break
example_label_day = _valid[5]
example_in_day = example_label_day - timedelta(days=1)
_files = goes_cdt_day(example_in_day)
print(f"CDT D-1 input {example_in_day}  ->  flood day {example_label_day}")
print(f"{len(_files)} frames; scan UTC:", [f"{_scan_dt(f):%H:%M}" for f in _files])

## 4. Pixel image stack — 7 engineered channels

For one CDT day (D−1), read the raw bands across the 8 frames and build the 7-channel pixel
stack: the 3 kept raw bands (b8, b10, b14), then 2 brightness-temperature differences
(d10-8 low-WV gradient, d11-14 cloud phase), then the 3-hourly Δb14 and the running maximum
cloud-top cooling. (b11 is read only to form d11-14, not stored as its own channel.)
Everything is physical (Kelvin) at this stage; standardization happens later, just before
caching the image.

In [ ]:
B_INDEX = {b: i for i, b in enumerate(RAW_BANDS)}


def read_bands(f):
    "Read RAW_BANDS from one GOES NetCDF -> (len(RAW_BANDS),H,W) physical (K), space NaN."
    out = np.empty((len(RAW_BANDS), IMG_H, IMG_W), np.float32)
    with netCDF4.Dataset(f) as nc:
        for i, b in enumerate(RAW_BANDS):
            out[i] = np.ma.filled(nc[f"CMI_C{b:02d}"][:], np.nan)
    return out


def build_pixel_stack(in_day):
    "(T,7,H,W) physical image stack for CDT day in_day, plus the files and scan times."
    files = goes_cdt_day(in_day)[:T_FRAMES]
    raw = np.stack([read_bands(f) for f in files])        # (T,len(RAW_BANDS),H,W)
    T = raw.shape[0]
    img = np.empty((T, N_IMG_CH, IMG_H, IMG_W), np.float32)
    k = 0
    for b in RAW_KEEP:                                     # raw bands kept as channels
        img[:, k] = raw[:, B_INDEX[b]]
        k += 1
    for a, b in DIFF_PAIRS:                                # brightness-temperature diffs
        img[:, k] = raw[:, B_INDEX[a]] - raw[:, B_INDEX[b]]
        k += 1
    b14 = raw[:, B_INDEX[14]]
    d14 = np.zeros_like(b14)
    d14[1:] = b14[1:] - b14[:-1]                           # 3-hourly Δb14 (frame 0 = 0)
    img[:, k] = d14
    k += 1
    cooling = np.maximum(-d14, 0.0)                        # +ve = cloud-top cooling
    img[:, k] = np.maximum.accumulate(cooling, axis=0)     # running max cooling so far
    return img, files, [_scan_dt(f) for f in files]


def downsample_img(img):
    "(T,C,H,W) physical -> (T,C,H//DS,W//DS) by block-mean (anti-aliased, NaN-safe)."
    T, C, H, W = img.shape
    ds = DOWNSAMPLE
    x = img.reshape(T, C, H // ds, ds, W // ds, ds)
    with warnings.catch_warnings():                        # all-NaN space blocks -> NaN
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(x, axis=(3, 5)).astype(np.float32)


_img, _f, _times = build_pixel_stack(example_in_day)
print(f"pixel stack {_img.shape}  {_img.dtype}  = {_img.nbytes/1e6:.0f} MB (full-res physical)")
for c in range(N_IMG_CH):
    a = _img[:, c]
    print(f"  ch{c:>2} {IMG_CH_NAMES[c]:>12}: min {np.nanmin(a):8.2f}  max {np.nanmax(a):8.2f}")

## 5. GOES per-cell features (masked pooled)

Pool the pixel stack onto the 50 km grid, per frame — but only the within-cell distribution
of the IR window band b14, which mean-pooling (and the model's CellPool) cannot recover:
b14_min (the coldest pixel = deepest convection in the cell), the cold-cloud fractions
(b14 < 235 K, b14 < 220 K), and the cold-cloud area change (Δ of the b14 < 235 K fraction
versus the previous frame). The per-channel means were dropped — they duplicate what the
image branch already computes. Result: (8, 4, R, C).

In [ ]:
GOES_CELL_NAMES = ["b14_min", "frac_b14<235", "frac_b14<220", "ccarea_change"]
N_GOES_CELL = len(GOES_CELL_NAMES)        # 4
B14_CH = IMG_CH_NAMES.index("b14")        # b14's slot in the 7-channel image stack


def goes_cell_features(img):
    "(T,7,H,W) physical -> (T,4,R,C) masked-pooled b14 distribution per cell."
    T = img.shape[0]
    out = np.zeros((T, N_GOES_CELL, GRID_R, GRID_C), np.float32)
    prev_ccf = None
    for t in range(T):
        b14 = img[t, B14_CH]
        out[t, 0] = pool_min(b14)                          # coldest pixel = deepest convection
        ccf = pool_frac(b14, CC235)
        out[t, 1] = ccf                                    # cold-cloud (b14<235) coverage
        out[t, 2] = pool_frac(b14, CC220)                  # deep-convection (b14<220) coverage
        out[t, 3] = 0.0 if prev_ccf is None else (ccf - prev_ccf)   # cold-cloud growth
        prev_ccf = ccf
    return out


_goes = goes_cell_features(_img)
print(f"GOES cell features {_goes.shape}  {_goes.dtype}")
print(f"  {N_GOES_CELL} features:", GOES_CELL_NAMES)

## 6. GLM lightning per-cell features

For each sample, load the GLM flash parquet(s) for the UTC dates the 8 frames span, assign
each flash to its nearest frame (a 3-hour bin) and to its 50 km cell (Albers integer
binning), and aggregate per cell per bin. GLM essentially carries one dimension of signal
(flash_count ≈ density ≈ hourly-rate ≈ n_locations, r > 0.95), so we keep just two
near-independent features: the flash count and the total flash energy (femtojoules, ×1e15,
float32-safe). The buggy area field and the redundant count-derivatives are dropped.
Result: (8, 2, R, C).

In [ ]:
import pyproj

GX0, GY0, GSTEP, GGR, GGC = grid_transform()
assert (GGR, GGC) == (GRID_R, GRID_C), "grid_transform disagrees with build_grid_cells"
_ALBERS = pyproj.Transformer.from_crs(4326, 5070, always_xy=True)
GLM_COLS = ["time_start", "lat", "lon", "energy"]
GLM_FEAT_NAMES = ["flash_count", "total_energy_fJ"]
N_GLM = len(GLM_FEAT_NAMES)               # 2
ENERGY_SCALE = 1e15                        # J -> femtojoules (readable, float32-safe)


def _flash_rc(lon, lat):
    "Map flash lon/lat -> (R, C, in_grid) on the 50 km grid via Albers integer binning."
    x, y = _ALBERS.transform(np.asarray(lon), np.asarray(lat))
    x, y = np.asarray(x), np.asarray(y)
    C = np.floor((x - GX0) / GSTEP)
    R = (GGR - 1) - np.floor((y - GY0) / GSTEP)
    ok = np.isfinite(x) & np.isfinite(y) & (C >= 0) & (C < GGC) & (R >= 0) & (R < GGR)
    return (np.where(ok, R, 0).astype(np.int64), np.where(ok, C, 0).astype(np.int64), ok)


def _load_glm(dates):
    dfs = []
    for d in dates:
        p = GLM_DIR / f"{d.year}" / f"glm_flashes_{d:%Y%m%d}.parquet"
        if p.exists():
            dfs.append(pd.read_parquet(p, columns=GLM_COLS))
    if not dfs:
        return None
    return pd.concat(dfs, ignore_index=True)


def glm_cell_features(frame_times):
    "(T,2,R,C) per-3h-bin GLM cell features: flash count + total flash energy."
    out = np.zeros((T_FRAMES, N_GLM, GRID_R, GRID_C), np.float32)
    g = _load_glm(sorted({pd.Timestamp(ft).date() for ft in frame_times}))
    if g is None or len(g) == 0:
        return out
    t = g["time_start"].to_numpy().astype("datetime64[s]")
    ft = np.array([np.datetime64(pd.Timestamp(x), "s") for x in frame_times])
    dmat = np.abs((t[:, None] - ft[None, :]).astype("timedelta64[s]").astype(np.float64))
    bin_idx = dmat.argmin(1)
    keep = dmat.min(1) <= 5400.0                            # within 1.5 h of a frame
    R, C, ok = _flash_rc(g["lon"].to_numpy(), g["lat"].to_numpy())
    keep &= ok
    bin_idx, R, C = bin_idx[keep], R[keep], C[keep]
    cell = R * GRID_C + C
    energy = g["energy"].to_numpy(np.float64)[keep] * ENERGY_SCALE
    nc = N_CELL
    for b in range(T_FRAMES):
        m = bin_idx == b
        if not m.any():
            continue
        cb, eb = cell[m], energy[m]
        cnt = np.bincount(cb, minlength=nc).astype(np.float64)
        ten = np.bincount(cb, weights=eb, minlength=nc)
        out[b, 0] = cnt.reshape(GRID_R, GRID_C).astype(np.float32)
        out[b, 1] = ten.reshape(GRID_R, GRID_C).astype(np.float32)
    return out


_glm = glm_cell_features(_times)
print(f"GLM cell features {_glm.shape}  {_glm.dtype}")
print(f"  {N_GLM} features:", GLM_FEAT_NAMES)
print(f"  example day total flashes on land: {_glm[:, 0].sum():.0f}")

## 7. Daily summaries

Collapse the 8 frames to one map per feature: the day's coldest b14 and coldest b8 (the two
strongest single signals — the daily min picks up the peak convection an individual frame
can miss) and the daily GLM flash count. Result: (3, R, C). (Daily min b8 uses the per-cell
mean b8 minimized across frames; the weak max_cooling / max_ccfrac and the redundant GLM
rate / hours-lit summaries were dropped.)

In [ ]:
DAILY_NAMES = ["min_b14", "min_b8", "glm_count"]
B8_CH = IMG_CH_NAMES.index("b8")          # b8's slot in the 7-channel image stack


def daily_summary(img, goes_cell, glm_cell):
    "(T,7,H,W) img + per-frame cell stacks -> (3,R,C) daily summary."
    out = np.zeros((len(DAILY_NAMES), GRID_R, GRID_C), np.float32)
    T = img.shape[0]
    b8_mean = np.stack([pool_means(img[t, B8_CH:B8_CH + 1])[0] for t in range(T)])
    with warnings.catch_warnings():                       # ocean cells are all-NaN by design
        warnings.simplefilter("ignore", RuntimeWarning)
        out[0] = np.nanmin(goes_cell[:, 0], axis=0)       # min b14 (b14_min over frames)
        out[1] = np.nanmin(b8_mean, axis=0)               # min b8 (cell-mean b8 over frames)
    out[2] = glm_cell[:, 0].sum(0)                        # daily flash count
    return np.nan_to_num(out)


_daily = daily_summary(_img, _goes, _glm)
print(f"daily summary {_daily.shape}  ->", DAILY_NAMES)

## 8. Image-channel standardization stats

The image stack is standardized per channel before caching so a CNN sees roughly zero-mean
/ unit-variance inputs. Stats are computed on the **downsampled** image (what actually gets
stored), sampled across a dozen days spanning 2019-2026, and cached to JSON. The per-cell
feature grids are left in physical units (Kelvin, fractions, counts) so the final summary is
interpretable and each model can normalize them as it likes.

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
if STATS_PATH.exists():
    cs = json.loads(STATS_PATH.read_text())
    print(f"loaded cached image-channel stats from {STATS_PATH.name}")
else:
    all_days = sorted(labels)                          # sample ~12 valid days across all years
    rng = np.random.default_rng(0)
    acc, n_used = [[] for _ in range(N_IMG_CH)], 0
    for i in rng.permutation(len(all_days)):
        d = all_days[i]
        if not valid_day(d):
            continue
        img = downsample_img(build_pixel_stack(d - timedelta(days=1))[0])   # stored res
        for c in range(N_IMG_CH):
            acc[c].append(img[:, c, ::2, ::2].ravel())
        n_used += 1
        if n_used >= 12:
            break
    cs = {IMG_CH_NAMES[c]: {"mean": float(np.nanmean(np.concatenate(acc[c]))),
                            "std": float(np.nanstd(np.concatenate(acc[c])) + 1e-6)}
          for c in range(N_IMG_CH)}
    STATS_PATH.write_text(json.dumps(cs, indent=2))
    print(f"computed image-channel stats from {n_used} days -> {STATS_PATH}")

IMG_MEAN = np.array([cs[n]["mean"] for n in IMG_CH_NAMES], np.float32)
IMG_STD = np.array([cs[n]["std"] for n in IMG_CH_NAMES], np.float32)
for n, m, s in zip(IMG_CH_NAMES, IMG_MEAN, IMG_STD):
    print(f"  {n:>12}: mean {m:8.3f}  std {s:7.3f}")

## 9. Assemble one sample

build_sample ties it together: the standardized image stack (float16), the three per-cell
feature grids (float32), the normalized lead time, and the label.

In [ ]:
def lead_times(times, label_day):
    "Per-frame whole hours before day D's CDT start (D 05:00 UTC)."
    d05 = datetime(label_day.year, label_day.month, label_day.day, 5)
    return np.array([round((d05 - tt).total_seconds() / 3600) for tt in times], np.float32)


def build_sample(in_day, label_day):
    img, files, times = build_pixel_stack(in_day)         # full-res physical
    goes = goes_cell_features(img)                         # pooled from FULL-res
    glm = glm_cell_features(times)
    daily = daily_summary(img, goes, glm)                 # from FULL-res
    lead = lead_times(times, label_day)
    img_ds = downsample_img(img)                           # /2 for storage (signal kept above)
    img_std = (img_ds - IMG_MEAN[None, :, None, None]) / IMG_STD[None, :, None, None]
    np.nan_to_num(img_std, copy=False)                    # space pixels -> 0 (= channel mean)
    return {"img": img_std.astype(np.float16),
            "goes": goes.astype(np.float32),
            "glm": glm.astype(np.float32),
            "daily": daily.astype(np.float32),
            "t": (lead / 24.0).astype(np.float32),        # normalized lead (hours/24)
            "y": labels[label_day].astype(np.uint8)}


t0 = time.perf_counter()
_s = build_sample(example_in_day, example_label_day)
print(f"one sample in {time.perf_counter()-t0:.1f}s:")
for k, v in _s.items():
    print(f"  {k:>5}: {tuple(v.shape)}  {v.dtype}  = {v.nbytes/1e6:.2f} MB")
print(f"  per-sample on disk ~= {sum(v.nbytes for v in _s.values())/1e6:.0f} MB "
      f"(img dominates)")

## 10. Sample index and temporal split

A sample is valid when all 8 GOES frames exist for CDT day D−1. We index every valid
(D−1 → D) pair across 2019-2026 and assign the **canonical temporal split** by the label
year: train ≤ 2024, validate 2025, test 2026. Temporal (not random) so the held-out years
are genuinely unseen — the honest setup for reported numbers.

In [ ]:
rows = []
for d_label in sorted(labels):                            # ~2.7k goes_cdt_day globs (HDD)
    d_in = d_label - timedelta(days=1)
    if len(goes_cdt_day(d_in)) == T_FRAMES:
        rows.append({"in_day": pd.Timestamp(d_in), "label_day": pd.Timestamp(d_label),
                     "n_pos": int(labels[d_label].sum())})
index = pd.DataFrame(rows)

yr = index["label_day"].dt.year
index["split"] = np.where(yr <= TRAIN_MAX, "train",
                          np.where(yr == VAL_YEAR, "val", "test"))

print(index["split"].value_counts().reindex(["train", "val", "test"]).to_string())
print(f"\ntotal valid samples {min(YEARS)}-{max(YEARS)}: {len(index)}")
print("\nsamples per year x split:")
print(index.groupby([yr.rename("year"), "split"]).size().unstack(fill_value=0).to_string())
index.head()

## 11. Inspect one random sample — full feature visualization

Before materializing, look at one randomly chosen valid sample end to end: the GOES image
channels (as stored, /2 resolution), the b14 frame sequence, every per-cell GOES / GLM /
daily feature on the 50 km grid, and the output flood label. This is the complete contents
of one cached sample. (Drawn at random from the convective season so GLM panels are non-empty.)

In [ ]:
summer = index[index["label_day"].dt.month.isin(range(5, 10))]
r = (summer if len(summer) else index).sample(1, random_state=7).iloc[0]
samp = build_sample(r["in_day"].date(), r["label_day"].date())
img, goes, glm = samp["img"].astype(np.float32), samp["goes"], samp["glm"]
daily, lead, y = samp["daily"], samp["t"], samp["y"]
ind, ld = r["in_day"].date(), r["label_day"].date()
print(f"sample: input CDT D-1 {ind}  ->  flood day {ld}")
print(f"  flooded land cells: {int(y[land_mask].sum())}  |  total GLM flashes: "
      f"{glm[:, 0].sum():.0f}  |  per-frame lead (h): {(lead*24).astype(int).tolist()}")


def _mland(a):
    return np.where(land_mask, a, np.nan)             # land-only for display


def show_cellgrid(arr, names, title, cmap="viridis"):
    n = len(names)
    cols = 6
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.0 * cols, 1.85 * rows))
    for k, ax in enumerate(np.atleast_1d(axes).flat):
        if k < n:
            im = ax.imshow(_mland(arr[k]), cmap=cmap)
            ax.set_title(names[k], fontsize=7)
            fig.colorbar(im, ax=ax, shrink=0.6)
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


# representative frame = most cold cloud, but skip frame 0 (its dt_b14_3h and maxcool_b14
# are identically zero by construction, so they would render as empty panels)
_ccf_pf = np.nansum(np.where(land_mask, goes[:, 1], 0.0), axis=(1, 2))
_ccf_pf[0] = -1.0
fsel = int(_ccf_pf.argmax())
gbin = int(glm[:, 0].sum((1, 2)).argmax())                                         # busiest GLM bin

# 1) GOES image channels (standardized, as stored at /2) at the representative frame
icols = 4
irows = int(np.ceil(N_IMG_CH / icols))
fig, axes = plt.subplots(irows, icols, figsize=(2.6 * icols, 2.4 * irows))
for k, ax in enumerate(np.atleast_1d(axes).flat):
    if k < N_IMG_CH:
        ax.imshow(img[fsel, k, ::3, ::3], cmap="gray_r")
        ax.set_title(IMG_CH_NAMES[k], fontsize=8)
    ax.axis("off")
fig.suptitle(f"1. GOES image channels (standardized, stored /2) - frame {fsel} "
             f"(lead {int(lead[fsel]*24)} h)", fontsize=12)
plt.tight_layout()
plt.show()

# 2) b14 (IR window) across all 8 frames
fig, axes = plt.subplots(1, T_FRAMES, figsize=(2.0 * T_FRAMES, 2.5))
for t in range(T_FRAMES):
    axes[t].imshow(img[t, B14_CH, ::3, ::3], cmap="gray_r")
    axes[t].set_title(f"-{int(lead[t]*24)}h", fontsize=8)
    axes[t].axis("off")
fig.suptitle("2. b14 IR-window sequence across the 8 CDT D-1 frames", fontsize=11)
plt.tight_layout()
plt.show()

# 3) GOES per-cell features (4) at the representative frame
show_cellgrid(goes[fsel], GOES_CELL_NAMES, f"3. GOES per-cell features - frame {fsel}")

# 4) GLM per-cell features (2) at the busiest 3h bin
show_cellgrid(glm[gbin], GLM_FEAT_NAMES, f"4. GLM per-cell features - 3h bin {gbin}", cmap="hot")

# 5) daily summary features (3)
show_cellgrid(daily, DAILY_NAMES, "5. Daily summary features")

# 6) the output label
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(_mland(np.full_like(y, 0.12, np.float32)), cmap="Greys", vmin=0, vmax=1)
ax.imshow(np.where(land_mask & (y > 0), 1.0, np.nan), cmap="Reds", vmin=0, vmax=1)
ax.set_title(f"6. Output label y - {ld}  ({int(y[land_mask].sum())} flooded cells)", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

## 12. Materialize the cache

Precompute every 2019-2026 sample to CACHE_DIR in parallel. Per sample we write six files:
the float16 image stack (img, /2 res), the three float32 per-cell grids (goes / glm /
daily), the lead time (t), and the label (y). Reads come from the GOES HDD (disk4), writes
go to the NVMe — different drives, no contention — so ~16 workers saturate the read side.
Set BUILD_CACHE = True to build all of 2019-2026 (**~274 GB**; the build takes a few hours,
HDD-read-bound); leave it False for a 2-sample smoke test. A progress line prints every
PROGRESS_EVERY completed samples (count, %, GB written, samples/min, elapsed, ETA).

The writer skips any day whose _img.npy already exists, so the build is resumable — rerun to
pick up where it stopped. (If you ever change the feature/resolution contract, delete
CACHE_DIR first so stale files are not mixed in.)

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

CACHE_DIR.mkdir(parents=True, exist_ok=True)
index.to_parquet(CACHE_DIR / "manifest.parquet")
print(f"wrote manifest ({len(index)} samples) -> {CACHE_DIR}")

BUILD_CACHE = True                  # <- True to build ALL 2019-2026 samples (~274 GB)
N_WORKERS = 16
PROGRESS_EVERY = 25                  # print a progress line every N completed samples


def _write_one(args):
    "Build+save one sample. Returns (day, nbytes, err): err=None ok, nbytes=0 cached."
    in_day, label_day = args
    stem = CACHE_DIR / f"{label_day:%Y%m%d}"
    day = f"{label_day:%Y%m%d}"
    if Path(f"{stem}_img.npy").exists():
        return (day, 0, None)                             # already cached (resumable)
    for attempt in range(2):                              # one retry (transient HDD/HDF reads)
        try:
            s = build_sample(in_day, label_day)
            np.save(f"{stem}_goes.npy", s["goes"])
            np.save(f"{stem}_glm.npy", s["glm"])
            np.save(f"{stem}_daily.npy", s["daily"])
            np.save(f"{stem}_t.npy", s["t"])
            np.save(f"{stem}_y.npy", s["y"])
            np.save(f"{stem}_img.npy", s["img"])          # img LAST: its presence = complete
            return (day, int(s["img"].nbytes), None)
        except Exception as e:                            # corrupt/missing GOES file etc.
            if attempt == 0:
                continue
            return (day, 0, f"{type(e).__name__}: {e}")


todo = list(zip(index["in_day"].dt.date, index["label_day"].dt.date))
if not BUILD_CACHE:
    todo = todo[:2]
    print(f"smoke test: writing {len(todo)} samples (set BUILD_CACHE=True for all)")
else:
    print(f"building {len(todo)} samples on {N_WORKERS} workers "
          f"(~{len(todo)*_s['img'].nbytes/1e9:.0f} GB, HDD-read-bound) ...", flush=True)

t0 = time.perf_counter()
n_total, done, built, gb, failed = len(todo), 0, 0, 0.0, []
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(_write_one, a) for a in todo]
    for fut in as_completed(futures):
        day, nbytes, err = fut.result()                   # never raises (caught in worker)
        done += 1
        if err:
            failed.append((day, err))
        else:
            built += nbytes > 0
            gb += nbytes / 1e9
        if done % PROGRESS_EVERY == 0 or done == n_total:
            el = time.perf_counter() - t0
            rate = done / el                                  # samples/sec
            eta = (n_total - done) / rate / 60 if rate else 0  # minutes
            print(f"  [{done:>4}/{n_total}] {done/n_total:4.0%}  "
                  f"built {built:>4}  failed {len(failed):>2}  {gb:6.1f} GB  "
                  f"{rate * 60:5.1f}/min  elapsed {el / 60:5.1f}m  eta {eta:5.1f}m",
                  flush=True)
print(f"done: {built} written, {done - built - len(failed)} cached, {len(failed)} failed "
      f"in {(time.perf_counter() - t0) / 60:.1f} min -> {CACHE_DIR}")
if failed:
    print(f"\n{len(failed)} samples failed (corrupt/missing GOES files). Re-download those "
          f"GOES days and rerun this cell (it resumes):")
    for day, err in failed[:25]:
        print(f"  {day}: {err}")
    if len(failed) > 25:
        print(f"  ... and {len(failed) - 25} more")

## 13. Summary — features and dataset

A full accounting of what the cache holds: every feature group with its shape and channel
names, then the dataset totals — date range, sample counts and split, and the flooded vs
non-flooded land-cell balance (the class-imbalance the model faces).

In [ ]:
man = pd.read_parquet(CACHE_DIR / "manifest.parquet")
y_files = sorted(CACHE_DIR.glob("*_y.npy"))

print("=" * 78)
print(f"UNIFIED CACHE  ({min(YEARS)}-{max(YEARS)})  ->  {CACHE_DIR}")
print("=" * 78)

groups = [
    ("img   (image stack, /2 res)", f"(8, {N_IMG_CH}, {IMG_H_DS}, {IMG_W_DS})", "f16", IMG_CH_NAMES),
    ("goes  (per-cell, per frame)", f"(8, {N_GOES_CELL}, {GRID_R}, {GRID_C})", "f32", GOES_CELL_NAMES),
    ("glm   (per-cell, per 3h bin)", f"(8, {N_GLM}, {GRID_R}, {GRID_C})", "f32", GLM_FEAT_NAMES),
    ("daily (per-cell summary)", f"({len(DAILY_NAMES)}, {GRID_R}, {GRID_C})", "f32", DAILY_NAMES),
    ("t     (lead time)", "(8,)", "f32", ["hours_before_D / 24"]),
    ("y     (label)", f"({GRID_R}, {GRID_C})", "u8", ["1 = observed flood (gs U storm)"]),
]
print("\nFEATURES")
for name, shape, dt, feats in groups:
    print(f"\n  {name}  shape {shape}  {dt}")
    for i in range(0, len(feats), 3):
        print("      " + ", ".join(feats[i:i + 3]))

print("\n" + "=" * 78)
print("DATASET")
print("=" * 78)
print(f"  date range (label day) : {man['label_day'].min().date()} -> {man['label_day'].max().date()}")
print(f"  total samples (days)   : {len(man)}")
sp = man['split'].value_counts().reindex(['train', 'val', 'test'])
print(f"  split (temporal)       : train {sp['train']} (<= {TRAIN_MAX})  "
      f"val {sp['val']} ({VAL_YEAR})  test {sp['test']} ({TEST_YEAR})")
print(f"  cached on disk so far  : {len(y_files)} samples")

if y_files:
    man = man.set_index(man["label_day"].dt.strftime("%Y%m%d"))
    rows = []
    for split in ["train", "val", "test"]:
        days = [d for d in man.index[man["split"] == split] if (CACHE_DIR / f"{d}_y.npy").exists()]
        if not days:
            continue
        Y = np.stack([np.load(CACHE_DIR / f"{d}_y.npy") for d in days]).astype(bool)
        ld = land_mask[None]
        per_day = (Y & ld).sum((1, 2))
        base = (Y & ld).sum() / (land_mask.sum() * len(Y))
        rows.append((split, len(days), per_day.mean(), int(per_day.max()), base))
    print(f"\n  {'split':<7}{'days':>6}{'mean pos/day':>14}{'max':>6}{'base rate':>12}")
    for s, n, mp, mx, br in rows:
        print(f"  {s:<7}{n:>6}{mp:>14.1f}{mx:>6}{br:>11.3%}")
    print(f"\n  land cells per day: {n_land}  |  positives are rare -> use the recall-favoring loss")
else:
    print("\n  (no samples cached yet - set BUILD_CACHE=True and run the materialize cell)")